# Multi-household baseline comparison

No-PV baseline against PV scenarios for several households, with separate and group
invoice views. No optimization and no VPP control: interval pricing under the SI
dynamic package, plus a souporaba monthly settlement summary.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from Data_Loader import load_multiple_households
from multi_household_tools import (
    run_interval_scenario,
    run_souporaba_monthly_scenario,
)

In [ ]:
# Select a small household group for baseline comparison.
HOUSEHOLD_IDS = [29, 30, 31]
HOUSEHOLD_DATA = {str(k): v for k, v in load_multiple_households(HOUSEHOLD_IDS).items()}

print(f"Loaded households: {list(HOUSEHOLD_DATA.keys())}")
for hid, df in HOUSEHOLD_DATA.items():
    print(f"{hid}: rows={len(df)} from {df.index.min()} to {df.index.max()}")

In [ ]:
# Scenario 1: no PV baseline under SI dynamic package (GEN-I).
baseline_no_pv = run_interval_scenario(
    HOUSEHOLD_DATA,
    scenario_name="GENI_DINAMICNI_no_pv",
    scheme="si_dobava",
    paket_id="GENI_DINAMICNI",
    pv_scale_map={hid: 0.0 for hid in HOUSEHOLD_DATA.keys()},
)

# Scenario 2: PV enabled under SI samooskrba dynamic package (GEN-I).
pv_enabled = run_interval_scenario(
    HOUSEHOLD_DATA,
    scenario_name="GENI_SAMO_DINAMICNI_pv",
    scheme="si_samooskrba",
    paket_id="GENI_SAMO_DINAMICNI",
    pv_scale_map={hid: 1.0 for hid in HOUSEHOLD_DATA.keys()},
)

In [ ]:
summary_compare = pd.concat([baseline_no_pv["summary"], pv_enabled["summary"]], ignore_index=True)
display(summary_compare)

group_only = summary_compare[summary_compare["household_id"] == "GROUP"].copy()
display(group_only[["scenario", "total_cost_eur", "total_imported_kwh", "total_exported_kwh"]])

In [ ]:
# Separate and group invoice line-item views from interval scenarios.
baseline_invoices = baseline_no_pv["invoice_views"]
pv_invoices = pv_enabled["invoice_views"]

print("Baseline GROUP invoice rows:")
display(pd.DataFrame(baseline_invoices["group"]).head(20))

print("PV GROUP invoice rows:")
display(pd.DataFrame(pv_invoices["group"]).head(20))

first_hh = list(HOUSEHOLD_DATA.keys())[0]
print(f"Baseline separate invoice rows for household {first_hh}:")
display(pd.DataFrame(baseline_invoices["separate"][first_hh]).head(20))

In [ ]:
# Souporaba monthly settlement summary:
# - first household is oddajnik
# - remaining households are prejemniki
oddajnik_id = list(HOUSEHOLD_DATA.keys())[0]
prejemnik_ids = list(HOUSEHOLD_DATA.keys())[1:]

souporaba_df = run_souporaba_monthly_scenario(
    HOUSEHOLD_DATA,
    scenario_name="GENI_SOUPORABA_dynamicni",
    oddajnik_id=oddajnik_id,
    prejemnik_ids=prejemnik_ids,
    pv_scale_map={hid: (1.0 if hid == oddajnik_id else 0.0) for hid in HOUSEHOLD_DATA.keys()},
    organizer_service_id="GENI_SOUPORABA",
    oddajnik_paket_id="GENI_SAMO_DINAMICNI",
    prejemnik_paket_id="GENI_DINAMICNI",
)

display(souporaba_df)

In [ ]:
# Quick visual for group total comparison.
group_plot_df = group_only[["scenario", "total_cost_eur"]].copy()
ax = group_plot_df.plot(kind="bar", x="scenario", y="total_cost_eur", legend=False, figsize=(8, 4))
ax.set_title("Primerjava skupnega stroška skupine")
ax.set_ylabel("Skupni strošek [EUR]")
plt.tight_layout()
plt.show()